# 02 – Extração de Citações via Google Scholar

## Resumo
Busca o número de citações de cada artigo do JMOe usando a API do **ScraperAPI**
para contornar os bloqueios do Google Scholar.

**Fluxo:**
1. Carrega o CSV de artigos (`artigos_jmoe_final.csv`).
2. Para cada artigo sem citação registrada, consulta o Google Scholar.
3. Extrai o número de citações via scraping do HTML retornado.
4. Salva progresso incrementalmente (checkpoint) em `artigos_atualizados.csv`.

**Entrada:** `files_csv/artigos_jmoe_final.csv`  
**Saída:** `files_csv/artigos_atualizados.csv`

> Substitua `'CHAVE_API_SCRAPERAPI'` pela sua chave real em https://scraperapi.com


## Instalação de Dependências

In [ ]:
# beautifulsoup4: parse de HTML | requests: requisições HTTP
!pip install beautifulsoup4 requests pandas -q


## Imports e Configurações

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import os

# ---------------------------------------------------------------------------
# CONFIGURAÇÕES – ajuste os caminhos e a chave antes de rodar
# ---------------------------------------------------------------------------
API_KEY      = 'CHAVE_API_SCRAPERAPI'          # Insira sua chave aqui
SCRAPER_URL  = 'http://api.scraperapi.com'

# Caminhos relativos à raiz do repositório
ARQUIVO_ENTRADA = 'files_csv/artigos_jmoe_final.csv'
ARQUIVO_SAIDA   = 'files_csv/artigos_atualizados.csv'


## Carregamento dos Dados com Checkpoint

In [ ]:
# Carrega o CSV original de artigos
df_original = pd.read_csv(ARQUIVO_ENTRADA)

# Sistema de checkpoint: se a saída já existe, retoma de onde parou
# Isso evita perder progresso em caso de interrupção
if os.path.exists(ARQUIVO_SAIDA):
    df = pd.read_csv(ARQUIVO_SAIDA)
    print(f'Retomando progresso. {len(df)} linhas carregadas.')
else:
    df = df_original.copy()
    df['Citações'] = None  # Inicializa coluna vazia na primeira execução
    print('Iniciando novo processamento.')


## Função de Busca no Google Scholar

In [ ]:
def get_citacoes_scholar(titulo):
    """
    Consulta o Google Scholar via ScraperAPI e retorna o número de citações.

    Parâmetros
    ----------
    titulo : str – título do artigo a ser buscado.

    Retorno
    -------
    int  – número de citações encontradas (0 se não encontrou).
    None – em caso de erro de conexão ou timeout.
    """
    # Ignora títulos inválidos (NaN, vazio)
    if not isinstance(titulo, str) or titulo.strip() == '':
        return 0

    payload = {
        'api_key':      API_KEY,
        'url':          f'https://scholar.google.com/scholar?q={titulo}',
        'country_code': 'us'
    }
    try:
        response = requests.get(SCRAPER_URL, params=payload, timeout=60)
        if response.status_code == 200:
            soup   = BeautifulSoup(response.text, 'html.parser')
            result = soup.find('div', {'class': 'gs_ri'})  # Primeiro resultado
            if result:
                # Busca o link 'Citado por N' ou 'Cited by N'
                link_citacao = result.find('a', string=re.compile(
                    r'Citado por|Cited by|\d+ citations'))
                if link_citacao:
                    num = re.findall(r'\d+', link_citacao.get_text())
                    return int(num[0]) if num else 0
            return 0
        return None
    except Exception:
        return None


## Loop de Processamento

In [ ]:
print('Iniciando busca de citações...')

for index, row in df.iterrows():
    # Processa apenas linhas sem citação registrada (NaN)
    if pd.isna(row['Citações']):
        titulo = row['titulo']

        # Trata títulos com valor NaN (tipo float)
        if not isinstance(titulo, str):
            print(f'[{index+1}] Pulando: Título inválido/vazio encontrado.')
            df.at[index, 'Citações'] = 0
            continue

        print(f'[{index+1}/{len(df)}] Buscando: {titulo[:60]}...')

        resultado = get_citacoes_scholar(titulo)

        if resultado is not None:
            df.at[index, 'Citações'] = resultado
            # Salva após cada artigo processado com sucesso (checkpoint)
            df.to_csv(ARQUIVO_SAIDA, index=False)
            print(f'   -> Citações: {resultado}')

        time.sleep(1)  # Pausa entre requisições para evitar bloqueio

print(f'\n PROCESSO FINALIZADO! Arquivo salvo em: {ARQUIVO_SAIDA}')
